这是一个未完善的TF-IDF demo。
用到了稀疏矩阵优化存储。

In [ ]:
import os
import numpy as np
import scipy.sparse as sp
from collections import Counter
from typing import List



def build_sparse_corpus(
    texts: List[str],
    tokenizer,
    max_length: int,
    special_token_ids: List[int],
    vocab_size: int,
):
    doc_term_counts: List[Counter] = []
    df_counts = Counter()
    total = len(texts)

    # ============================
    # 1. 统计 TF 和 DF
    # ============================
    for idx, text in enumerate(texts):
        token_ids = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            add_special_tokens=True,
        )["input_ids"]

        token_ids = [t for t in token_ids if t not in special_token_ids]

        counter = Counter(token_ids)
        doc_term_counts.append(counter)

        for token_id in counter.keys():
            df_counts[token_id] += 1

        if (idx + 1) % 200 == 0 or idx + 1 == total:
            print(f"\r  [sparse] {idx+1}/{total} ({(idx+1)*100//total}%)", end="", flush=True)
    print()

    # ============================
    # 2. 计算 IDF
    # ============================
    n_docs = len(texts)
    idf = np.zeros(vocab_size, dtype=np.float32)

    for token_id, df in df_counts.items():
        idf[token_id] = np.log((1 + n_docs) / (1 + df)) + 1.0

    # ============================
    # 3. 构建 CSR 稀疏矩阵
    # ============================
    rows, cols, vals = [], [], []

    for row_idx, counter in enumerate(doc_term_counts):
        for token_id, tf in counter.items():
            rows.append(row_idx)
            cols.append(int(token_id))
            vals.append(float(tf * idf[token_id]))

    if not rows:
        return sp.csr_matrix((n_docs, vocab_size), dtype=np.float32), idf

    sparse_matrix = sp.csr_matrix(
        (np.array(vals, dtype=np.float32), (np.array(rows), np.array(cols))),
        shape=(n_docs, vocab_size),
        dtype=np.float32,
    )

    return sparse_matrix, idf




from transformers import AutoTokenizer


from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-chinese",
    local_files_only=True,
    use_fast=False
)




special_token_ids = [
    tokenizer.cls_token_id,
    tokenizer.sep_token_id,
    tokenizer.pad_token_id,
]

vocab_size = tokenizer.vocab_size

texts = [
    "苹果公司发布了新的手机",
    "小米公司也发布了新的手机",
    "苹果做的馅饼是很好吃的"
]

# ====== 调用函数 ======
sparse_matrix, idf = build_sparse_corpus(
    texts=texts,
    tokenizer=tokenizer,
    max_length=128,
    special_token_ids=special_token_ids,
    vocab_size=vocab_size,
)

print("稀疏矩阵形状:", sparse_matrix.shape)
print("非零元素数量:", sparse_matrix.nnz)
print("示例前 10 个 IDF:", idf[:10])

